## 1: Environment Setup and Library Imports

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller
import yfinance as yf

# Plot styling configuration
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

## 2: Two-Year Historical Data Ingestion & Caching

In [ ]:
# Define liquid asset universe across sectors
tickers = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "NVDA",
    "META",
    "TSLA",
    "AMD",
    "JPM",
    "BAC",
]
start_date = "2023-01-01"
end_date = "2025-01-01"

# Ingest adjusted closing prices and compute discrete daily percentage returns
raw_prices = yf.download(tickers, start=start_date, end=end_date)["Close"]
returns = raw_prices.pct_change().dropna()

print(f"Ingested Returns Matrix: {returns.shape[0]} trading days x {returns.shape[1]} assets")
returns.head()

## 3: PCA Factor Decomposition (Systematic Risk Extraction)

In [ ]:
# 1. Standardize (mean-center) returns
X = (returns - returns.mean()).values
T, N = X.shape
K = 3  # Retain top 3 systematic eigenfactors

# 2. Sample Covariance Matrix: Sigma = (1 / (T - 1)) * X^T @ X
cov_matrix = np.dot(X.T, X) / (T - 1)

# 3. Eigendecomposition: Sigma * v = lambda * v
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# 4. Project into factor return series: F = X @ V_top
top_eigenvectors = eigenvectors[:, :K]
factor_returns = pd.DataFrame(
    np.dot(X, top_eigenvectors),
    index=returns.index,
    columns=[f"PC_{i+1}" for i in range(K)],
)

explained_variance = eigenvalues[:K] / np.sum(eigenvalues)
print(f"Top {K} Components Cumulative Explained Variance: {explained_variance.sum():.2%}")

## 4: Residual Extraction Engines — Kalman Filter vs. Static OLS

In [ ]:
class KalmanResidualFilter:
    """Recursive state-space Bayesian filter for dynamic factor loading estimation."""

    def __init__(
        self,
        n_factors: int,
        process_noise: float = 1e-4,
        measurement_noise: float = 1e-3,
    ):
        self.dim = n_factors + 1  # Intercept (alpha) + K factor betas
        self.Q = np.eye(self.dim) * process_noise
        self.R = measurement_noise

    def filter_series(
        self, y: np.ndarray, F: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray]:
        T_steps = len(y)
        beta_hat = np.zeros(self.dim)
        P = np.eye(self.dim) * 1.0

        innovations = np.zeros(T_steps)
        betas_history = np.zeros((T_steps, self.dim))

        for t in range(T_steps):
            H_t = np.insert(F[t], 0, 1.0)  # Design vector: [1, F1_t, ..., FK_t]

            # 1. Predict
            beta_pred = beta_hat
            P_pred = P + self.Q

            # 2. Innovation (Residual)
            y_pred = np.dot(H_t, beta_pred)
            e_t = y[t] - y_pred
            innovations[t] = e_t

            S_t = np.dot(H_t, np.dot(P_pred, H_t)) + self.R

            # 3. Measurement Update
            K_gain = np.dot(P_pred, H_t) / S_t
            beta_hat = beta_pred + K_gain * e_t
            P = P_pred - np.outer(K_gain, H_t) @ P_pred
            betas_history[t] = beta_hat

        return innovations, betas_history


# Target test asset
target_asset = "NVDA"
y_target = returns[target_asset].values
F_matrix = factor_returns.values

# --- Run Kalman Filter ---
kf = KalmanResidualFilter(
    n_factors=K, process_noise=1e-4, measurement_noise=1e-3
)
kf_residuals, kf_betas = kf.filter_series(y_target, F_matrix)

# --- Run Static OLS Baseline ---
F_design = np.column_stack([np.ones(T), F_matrix])
ols_betas, _, _, _ = np.linalg.lstsq(F_design, y_target, rcond=None)
ols_residuals = y_target - np.dot(F_design, ols_betas)

# Assemble DataFrames (with 30-day burn-in truncation)
burn_in = 30
dates = returns.index[burn_in:]

df_compare = pd.DataFrame(
    {
        "KF_Residual": kf_residuals[burn_in:],
        "OLS_Residual": ols_residuals[burn_in:],
        "KF_Cumulative_Spread": np.cumsum(kf_residuals[burn_in:]),
        "OLS_Cumulative_Spread": np.cumsum(ols_residuals[burn_in:]),
    },
    index=dates,
)

## 5: Diagnostic Plot 1 — Dynamic Factor Beta Drift Over 2 Years

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

feature_labels = ["Alpha (Intercept)", "Beta PC_1 (Market)", "Beta PC_2 (Sector)"]
for i in range(3):
    axes[i].plot(
        returns.index[burn_in:],
        kf_betas[burn_in:, i],
        color="crimson",
        lw=1.8,
        label=f"Kalman Filter Time-Varying {feature_labels[i]}",
    )
    axes[i].axhline(
        ols_betas[i],
        color="black",
        ls="--",
        lw=1.5,
        label=f"Static OLS Baseline: {ols_betas[i]:.4f}",
    )
    axes[i].set_ylabel("Loading")
    axes[i].legend(loc="upper left")
    axes[i].set_title(f"{target_asset} Dynamic Factor Exposure: {feature_labels[i]}")

plt.tight_layout()
plt.show()

## 6: Diagnostic Plot 2 — Cumulative Idiosyncratic Spread Tracking

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(
    df_compare["KF_Cumulative_Spread"],
    color="navy",
    lw=1.8,
    label="Kalman Filter Spread (Dynamic)",
)
plt.plot(
    df_compare["OLS_Cumulative_Spread"],
    color="darkorange",
    lw=1.5,
    ls="--",
    label="OLS Spread (Static)",
)
plt.title(f"{target_asset} Cumulative Idiosyncratic Residual Spread (2-Year Horizon)")
plt.ylabel("Cumulative Innovation Spread")
plt.xlabel("Date")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 7: Statistical Stationarity & Autocorrelation Verification

In [ ]:
def evaluate_spread_properties(series: pd.Series, name: str):
    res = adfuller(series, autolag="AIC")
    lag1_autocorr = series.diff().dropna().autocorr(lag=1)
    print(f"=== {name} Diagnostics ===")
    print(f"  ADF Statistic:        {res[0]:.4f}")
    print(f"  ADF p-value:          {res[1]:.4e} ({'Stationary (p < 0.05)' if res[1] < 0.05 else 'Non-Stationary'})")
    print(f"  Critical Value (5%):  {res[4]['5%']:.4f}")
    print(f"  Lag-1 Autocorrelation: {lag1_autocorr:.4f} ({'Mean-Reverting' if lag1_autocorr < 0 else 'Momentum'})\n")

evaluate_spread_properties(df_compare["KF_Cumulative_Spread"], "Kalman Filter Spread")
evaluate_spread_properties(df_compare["OLS_Cumulative_Spread"], "Static OLS Spread")

## 8: Ornstein-Uhlenbeck Calibration on Kalman Spread

In [ ]:
# Fit AR(1): x_n = a + b * x_{n-1} + zeta
x = df_compare["KF_Cumulative_Spread"].values
x_lag = x[:-1]
x_curr = x[1:]

design = np.column_stack([np.ones(len(x_lag)), x_lag])
params, _, _, _ = np.linalg.lstsq(design, x_curr, rcond=None)
a_hat, b_hat = params[0], params[1]

dt = 1.0 / 252.0
kappa_ou = -np.log(b_hat) / dt
theta_ou = a_hat / (1.0 - b_hat)
var_zeta = np.var(x_curr - (a_hat + b_hat * x_lag), ddof=2)
sigma_eq = np.sqrt(var_zeta / (1.0 - b_hat**2))
# kappa is annualized (dt = 1/252), so ln(2)/kappa is in YEARS.
# The half-life in trading days is ln(2) / -ln(b).
half_life_days = np.log(2.0) / (-np.log(b_hat))

# Compute standardized s-score series
df_compare["s_score"] = (df_compare["KF_Cumulative_Spread"] - theta_ou) / sigma_eq

print("=== Continuous-Time OU Calibration (KF Spread) ===")
print(f"  Mean-Reversion Speed (kappa): {kappa_ou:.2f}")
print(f"  Equilibrium Center (theta):    {theta_ou:.4f}")
print(f"  Stationary Vol (sigma_eq):     {sigma_eq:.4f}")
print(f"  Half-Life of Mean Reversion:  {half_life_days:.2f} trading days")